# Examen ICCD753 Recuperación de Información

 Harry Guajan
 11/18/2026
## 1. Objetivo

Desarrollar un sistema de recuperación de información sobre un corpus asignado. El sistema deberá indexar los documentos, recuperar y ordenar resultados mediante TF-IDF con similitud coseno y BM25, y exponer la búsqueda como un servicio web publicado en internet.

Se evaluará el servicio enviando consultas desde un cliente (p.ej. Postman) y contrastando los resultados con los juicios de relevancia (*qrels*) del corpus.

## 2. Corpus asignado



| Corpus | Documentos y consultas | Qrels |
| --- | --- | --- |
| NFCorpus | [BeIR/nfcorpus](https://huggingface.co/datasets/BeIR/nfcorpus) | [BeIR/nfcorpus-qrels](https://huggingface.co/datasets/BeIR/nfcorpus-qrels) |


## 3. Preparación e indexación del corpus NFCorpus

La unidad de indexación será cada documento completo, identificándolo por `_id`. No se dividirán los textos en fragmentos.

Este bloque de código básicamente se encarga de preparar y validar toda la base de datos antes de arrancar. Primero importa las librerías necesarias y se asegura de que estén descargadas las *stopwords* de NLTK para no tener errores más adelante.  luego busca el archivo `.parquet` dentro de la carpeta del corpus y lo carga en una tabla. Una vez cargado, revisa que no falten las columnas principales (`_id`, `title` y `text`), limpia los datos pasando todo a texto, rellenando los vacíos para que no se rompa el programa y verificando que no haya IDs duplicados, para finalmente imprimir el total de documentos y mostrar las primeras filas como confirmación de que todo está listo.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import re
import pandas as pd
import nltk
from IPython.display import display
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer

for recurso in ("stopwords",):
    try:
        nltk.data.find(f"corpora/{recurso}")
    except LookupError:
        nltk.download(recurso, quiet=True)

candidatos = [
    Path("corpus"),
    Path("EXAMENRI2BTAS") / "corpus",
]
CORPUS_DIR = next((ruta for ruta in candidatos if ruta.is_dir()), candidatos[0])
corpus_archivos = list(CORPUS_DIR.glob("*.parquet"))
if len(corpus_archivos) != 1:
    raise RuntimeError(f"Se esperaba un archivo Parquet en {CORPUS_DIR}, se encontraron {len(corpus_archivos)}.")

corpus = pd.read_parquet(corpus_archivos[0])
requeridas = {"_id", "title", "text"}
faltantes = requeridas.difference(corpus.columns)
if faltantes:
    raise KeyError(f"Faltan columnas obligatorias: {sorted(faltantes)}")

corpus = corpus[["_id", "title", "text"]].copy()
corpus["_id"] = corpus["_id"].astype(str)
corpus["title"] = corpus["title"].fillna("").astype(str)
corpus["text"] = corpus["text"].fillna("").astype(str)

if corpus["_id"].duplicated().any():
    duplicados = corpus.loc[corpus["_id"].duplicated(), "_id"].tolist()
    raise ValueError(f"Hay identificadores duplicados: {duplicados[:10]}")

print(f"Corpus: {len(corpus)} documentos")
display(corpus.head())

Corpus: 3633 documentos


,_id,title,text
0,MED-10,Statin Use and Breast Cancer Survival: A Natio...,"Recent studies have suggested that statins, an..."
1,MED-14,Statin use after diagnosis of breast cancer an...,BACKGROUND: Preclinical studies have shown tha...
2,MED-118,Alkylphenols in human milk and their relations...,The aims of this study were to determine the c...
3,MED-301,Methylmercury: A Potential Environmental Risk ...,Epilepsy or seizure disorder is one of the mos...
4,MED-306,Sensitivity of Continuous Performance Test (CP...,Hit Reaction Time latencies (HRT) in the Conti...


### Decisiones de preprocesamiento

- **Normalización:** conversión a minúsculas.
- **Tokenización:** expresiones regulares sobre palabras alfanuméricas; se conservan palabras de al menos dos caracteres.
- **Stopwords:** eliminación con la lista inglesa de NLTK.
- **Stemming:** Snowball Stemmer para inglés.
- **Texto indexado:** título y texto completo del documento, sin fragmentación.

Este bloque hace   limpieza del texto y arma las estructuras principales del buscador para poder hacer consultas después. Primero une el título y el contenido de cada documento y los pasa por una función de preprocesamiento que convierte todo a minúsculas, saca palabras de al menos dos caracteres, elimina las *stopwords* en inglés y reduce cada término a su raíz usando *stemming*. 

Con esos textos ya limpios, construye el vocabulario completo y arma el índice invertido  además  calcula los pesos TF-IDF de todo el corpus en una matriz numérica lista para reutilizar. Al final, imprime estadísticas generales como el tamaño del vocabulario y la longitud promedio de los documentos, y muestra dos tablas de resumen para verificar que tanto los textos procesados como el índice de términos quedaron bien armados.

In [2]:
stopwords_ingles = set(stopwords.words("english"))
stemmer = SnowballStemmer("english")
patron_tokens = re.compile(r"(?u)\b[a-z0-9][a-z0-9]+\b")

def preprocesar(texto):
    tokens = patron_tokens.findall(str(texto).lower())
    return [stemmer.stem(token) for token in tokens if token not in stopwords_ingles]

corpus["search_text"] = (
    corpus["title"].str.strip() + "\n" + corpus["text"].str.strip()
).str.strip()
corpus["tokens"] = corpus["search_text"].map(preprocesar)
corpus["processed_text"] = corpus["tokens"].map(lambda tokens: " ".join(tokens))

document_ids = corpus["_id"].tolist()
titles = corpus["title"].tolist()
original_texts = corpus["text"].tolist()
processed_texts = corpus["processed_text"].tolist()
token_lists = corpus["tokens"].tolist()

document_lengths = [len(tokens) for tokens in token_lists]
vocabulario = sorted({termino for tokens in token_lists for termino in set(tokens)})

# Índice invertido: termino -> [(posicion_documento, frecuencia)]
indice_invertido = defaultdict(list)
for doc_pos, tokens in enumerate(token_lists):
    frecuencias = Counter(tokens)
    for termino, frecuencia in frecuencias.items():
        indice_invertido[termino].append((doc_pos, frecuencia))

df = {termino: len(postings) for termino, postings in indice_invertido.items()}

# Relación estable entre cada posición interna y el identificador original.
id_por_posicion = {pos: doc_id for pos, doc_id in enumerate(document_ids)}
posicion_por_id = {doc_id: pos for pos, doc_id in id_por_posicion.items()}

# Representación TF-IDF para reutilizarla en el siguiente paso.
tfidf_vectorizer = TfidfVectorizer(
    lowercase=False,
    use_idf=True,
    smooth_idf=True,
    sublinear_tf=False,
    norm="l2",
)
tfidf_matrix = tfidf_vectorizer.fit_transform(processed_texts)

print("Documentos:", len(document_ids))
print("Vocabulario:", len(vocabulario))
print("Términos indexados:", len(indice_invertido))
print("Longitud mínima/media/máxima:",
      min(document_lengths),
      round(sum(document_lengths) / len(document_lengths), 2),
      max(document_lengths))
print("Forma de la matriz TF-IDF:", tfidf_matrix.shape)

tabla_corpus = pd.DataFrame({
    "Identificador": document_ids,
    "Título": titles,
    "Texto": original_texts,
    "Términos procesados": [len(tokens) for tokens in token_lists],
})
display(tabla_corpus.head(10))

tabla_vocabulario = pd.DataFrame([
    {"término": termino, "df": len(postings)}
    for termino, postings in sorted(indice_invertido.items())
])
display(tabla_vocabulario.head(20))

Documentos: 3633
Vocabulario: 18807
Términos indexados: 18807
Longitud mínima/media/máxima: 11 155.23 983
Forma de la matriz TF-IDF: (3633, 18807)


,Identificador,Título,Texto,Términos procesados
0,MED-10,Statin Use and Breast Cancer Survival: A Natio...,"Recent studies have suggested that statins, an...",193
1,MED-14,Statin use after diagnosis of breast cancer an...,BACKGROUND: Preclinical studies have shown tha...,181
2,MED-118,Alkylphenols in human milk and their relations...,The aims of this study were to determine the c...,119
3,MED-301,Methylmercury: A Potential Environmental Risk ...,Epilepsy or seizure disorder is one of the mos...,125
4,MED-306,Sensitivity of Continuous Performance Test (CP...,Hit Reaction Time latencies (HRT) in the Conti...,188
5,MED-329,Phosphate is a vascular toxin.,Elevated phosphate (P) levels are seen in adva...,165
6,MED-330,Dietary Phosphorus Acutely Impairs Endothelial...,Excessive dietary phosphorus may increase card...,144
7,MED-332,Public health impact of dietary phosphorus exc...,This review explores the potential adverse imp...,178
8,MED-334,Differences among total and in vitro digestibl...,"OBJECTIVE: Among plant foods, grain products, ...",187
9,MED-335,Differences among total and in vitro digestibl...,OBJECTIVE: Meat and milk products are importan...,181


,término,df
0,00,91
1,000,136
2,000006073,1
3,00001,4
4,0001,117
5,0002,18
6,0003,9
7,0004,3
8,0005,8
9,0006,3


## 3.2. Modelos de recuperación

Se implementarán dos rankings independientes:

- **TF-IDF + similitud coseno:** compara el vector TF-IDF normalizado de la consulta con los vectores normalizados de los documentos.
- **BM25:** utiliza frecuencias documentales, frecuencia del término en el documento y normalización de longitud con $k_1=1.2$ y $b=0.75$.

Ambos métodos reciben texto libre, conservan los identificadores originales y pueden limitar el ranking mediante `top_k`.

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

BM25_K1 = 1.2
BM25_B = 0.75
longitud_media = sum(document_lengths) / len(document_lengths)

def preparar_resultados(puntuaciones, top_k=10):
    """Ordena puntuaciones, elimina posiciones duplicadas y devuelve los mejores documentos."""
    top_k = max(1, min(int(top_k), len(document_ids)))
    orden = np.argsort(-np.asarray(puntuaciones, dtype=float), kind="stable")

    resultados = []
    ids_vistos = set()
    for doc_pos in orden:
        doc_id = document_ids[int(doc_pos)]
        if doc_id in ids_vistos:
            continue
        ids_vistos.add(doc_id)
        resultados.append({
            "rank": len(resultados) + 1,
            "document_id": doc_id,
            "score": float(puntuaciones[int(doc_pos)]),
            "title": titles[int(doc_pos)],
            "snippet": original_texts[int(doc_pos)][:300].strip(),
        })
        if len(resultados) == top_k:
            break
    return resultados

def buscar_tfidf(query, top_k=10):
    tokens = preprocesar(query)
    processed_query = " ".join(tokens)
    query_vector = tfidf_vectorizer.transform([processed_query])
    puntuaciones = cosine_similarity(query_vector, tfidf_matrix).ravel()
    return preparar_resultados(puntuaciones, top_k)

def buscar_bm25(query, top_k=10, k1=BM25_K1, b=BM25_B):
    consulta_tokens = preprocesar(query)
    puntuaciones = np.zeros(len(document_ids), dtype=float)
    total_documentos = len(document_ids)

    for termino in set(consulta_tokens):
        postings = indice_invertido.get(termino, [])
        if not postings:
            continue

        frecuencia_documental = len(postings)
        idf = np.log(1 + (total_documentos - frecuencia_documental + 0.5) /
                     (frecuencia_documental + 0.5))

        for doc_pos, frecuencia in postings:
            denominador = frecuencia + k1 * (
                1 - b + b * document_lengths[doc_pos] / longitud_media
            )
            aporte = idf * (frecuencia * (k1 + 1)) / denominador
            puntuaciones[doc_pos] += aporte

    return preparar_resultados(puntuaciones, top_k)

print("Funciones TF-IDF y BM25 listas.")

### Prueba reproducible de los rankings

La siguiente consulta pertenece a las consultas disponibles en el parquet de NFCorpus. Se muestran 10 resultados independientes para TF-IDF y BM25.

In [ ]:
queries_candidatos = []
for ruta_queries in [Path("queries"), Path("EXAMENRI2BTAS") / "queries"]:
    if ruta_queries.is_dir():
        queries_candidatos = list(ruta_queries.glob("*.parquet"))
        break

if len(queries_candidatos) != 1:
    raise RuntimeError("No se encontró exactamente un archivo Parquet de consultas.")

queries = pd.read_parquet(queries_candidatos[0])
consulta_prueba = str(queries.iloc[0]["text"])
print("Consulta:", consulta_prueba)

ranking_tfidf = buscar_tfidf(consulta_prueba, top_k=10)
ranking_bm25 = buscar_bm25(consulta_prueba, top_k=10)

display(pd.DataFrame(ranking_tfidf).drop(columns="snippet"))
display(pd.DataFrame(ranking_bm25).drop(columns="snippet"))